# 按固定步骤拆分复杂问题

复杂问题往往需要几页资料。直接检索整句话时，多个要求会互相干扰。可以先按事先确认的步骤逐层拆开，分别检索最末端的问题，再合并实际返回的页面。

本页固定一棵可以逐项检查的拆分树，用来验证拆开后能否找回完整资料。它不代表系统已经能够自动写出可靠的子问题；自动生成子问题见下一页。

In [1]:
import sys
from pathlib import Path

def find_course_root(start):
    for folder in (start, *start.parents):
        if (folder / 'data' / 'dataset/manifest.json').is_file():
            return folder
    raise FileNotFoundError('没有找到教程数据目录，请从本节所在目录运行。')

course_root = find_course_root(Path.cwd())
if str(course_root) not in sys.path:
    sys.path.insert(0, str(course_root))

from common.eval_utils import build_bm25_search, load_query_catalog, load_pdf_pages
from common.nontraining_utils import load_annotation

data = load_query_catalog()
cases = {item['id']: item for item in data}
search = build_bm25_search(load_pdf_pages())

def rank_and_pages(results, expected_pages):
    expected = set(expected_pages)
    rank = next((index for index, item in enumerate(results, 1) if item.page in expected), None)
    return [item.page for item in results], rank

def leaves(question, tree):
    children = tree.get(question)
    if not children:
        return [question]
    return [leaf for child in children for leaf in leaves(child, tree)]

def retrieve_tree(root_question, tree):
    leaf_results = []
    for question in leaves(root_question, tree):
        leaf_results.append((question, search(question, top_k=1)[0]))
    seen, merged = set(), []
    for _, item in leaf_results:
        if item.page not in seen:
            seen.add(item.page)
            merged.append(item)
    return leaf_results, merged

main = cases['lda_derivation_recursive_steps']
main_tree = {
    main['query']: ['先明确投影目标', '再完成多分类求解'],
    '先明确投影目标': ['LDA 如何让同类投影接近、异类投影远离？'],
    '再完成多分类求解': [
        'LDA 的多分类优化目标怎样由二分类形式推广？',
        'LDA 如何用拉格朗日乘子得到 SbW 等于 SwWΛ？',
        'LDA 存在约束时为什么取最大的广义特征值？',
    ],
}
main_before = search(main['query'], top_k=5)
main_leaves, main_after = retrieve_tree(main['query'], main_tree)
main_annotation = load_annotation(main['id'])
before_pages, before_rank = rank_and_pages(main_before, main_annotation['expected_pages'])
after_pages, after_rank = rank_and_pages(main_after, main_annotation['expected_pages'])
coverage = sorted(set(after_pages).intersection(main_annotation['expected_pages']))
extra_pages = sorted(set(after_pages) - set(main_annotation['expected_pages']))
print('主要问题：', main['query'])
print('原问题前 5 页：', before_pages, '；必要页排名：', before_rank or '未出现')
print('拆分后的检索：')
for question, item in main_leaves:
    print(' -', question, '→ 第', item.page, '页')
print('合并页面：', after_pages, '；找到必要页：', coverage, '；额外页面：', extra_pages)
print('检索次数：1 →', len(main_leaves))
assert before_rank is None and coverage == [41, 42, 43, 44] and not extra_pages

check = cases['linear_regression_global_optimum']
check_tree = {check['query']: ['XTX 正定时平方损失是严格凸函数吗？', '正规方程 XTX w 等于什么？']}
check_before = search(check['query'], top_k=5)
check_leaves, check_after = retrieve_tree(check['query'], check_tree)
check_annotation = load_annotation(check['id'])
check_before_pages, check_before_rank = rank_and_pages(check_before, check_annotation['expected_pages'])
check_after_pages, check_after_rank = rank_and_pages(check_after, check_annotation['expected_pages'])
print('\n复查问题：', check['query'])
print('原问题前 5 页：', check_before_pages, '；必要页排名：', check_before_rank)
print('拆分后页面：', check_after_pages, '；必要页排名：', check_after_rank)
assert check_before_rank == 1 and check_after_rank is not None


主要问题： LDA 的多分类推导从投影目标到广义特征值要经过哪些步骤？
原问题前 5 页： [189, 126, 131, 132, 151] ；必要页排名： 未出现
拆分后的检索：
 - LDA 如何让同类投影接近、异类投影远离？ → 第 41 页
 - LDA 的多分类优化目标怎样由二分类形式推广？ → 第 42 页
 - LDA 如何用拉格朗日乘子得到 SbW 等于 SwWΛ？ → 第 43 页
 - LDA 存在约束时为什么取最大的广义特征值？ → 第 44 页
合并页面： [41, 42, 43, 44] ；找到必要页： [41, 42, 43, 44] ；额外页面： []
检索次数：1 → 4

复查问题： 在线性回归中，XTX 正定时为什么能求到全局最优解？
原问题前 5 页： [36, 58, 34, 45, 26] ；必要页排名： 1
拆分后页面： [36] ；必要页排名： 1


整句检索的前 5 页没有第 41～44 页。按投影目标、多分类推广、拉格朗日求解和特征值选择拆开后，四次检索分别找回第 41、42、43、44 页，没有混入额外页面。代价是检索次数从 1 次增加到 4 次。线性回归复查题原本已把第 36 页排在第 1，拆开后仍找到第 36 页。

这组结果只证明这棵固定拆分树有效。若让模型自动拆分，还要检查生成的问题有没有漏条件或偏离原问题。



## 递归分解和迭代补查不是一回事

Recursive Decomposition（递归分解）在检索前先把复合问题拆成互补的子问题，再分别检索和回答，再合并。它适合“问题本身就包含多个独立要求”；如果问题是单一事实，拆题只会增加调用次数和合并风险。

本页使用固定、人工检查过的拆分树。先看“LDA 从投影分离目标怎样推到 N−1 个最大广义特征值及其特征向量？”：从投影目标、拉格朗日关系到广义特征值逐层查找；再用“在线性回归中，XTX 正定时为什么能求到全局最优解？”确认原本已命中的问题没有被拆坏。固定树有效不等于任意问题都能这样拆；自动拆题见下一页。


In [2]:
# 实现要点：固定拆分树的最小执行步骤。
def recursive_retrieve(root_question, split_tree, search_fn, top_k=2):
    leaves = split_tree.get(root_question, [root_question])
    trace, evidence = [{"step": "decompose", "sub_questions": leaves}], []
    for index, sub_question in enumerate(leaves, start=1):
        hits = list(search_fn(sub_question, top_k=top_k)); evidence.extend(hits)
        trace.append({"step": "sub_retrieve", "index": index, "question": sub_question, "n_hits": len(hits)})
    trace.append({"step": "merge", "n_sub_questions": len(leaves)})
    return evidence, trace

# 实际回答阶段还应保留每个子问题的答案和来源，避免合并时丢失限定条件。


## Recursive Decomposition 的流程、适用条件与代码要点

Recursive Decomposition（递归分解）在检索前把复合问题拆成互补子问题，分别检索并回答，再按原问题合并。它不是“再搜一次”：决定发生在第一次检索之前，子问题应覆盖不同要求而不是把一句话拆成互相重复的碎片。

```text
root question → decompose
                 ├─ sub-question 1 → retrieve → sub-answer 1
                 ├─ sub-question 2 → retrieve → sub-answer 2
                 └─ sub-question 3 → retrieve → sub-answer 3
                                      ↓
                                  merge → answer
```

在 LDA 推导问题上，固定树从投影目标、推广关系、拉格朗日求解到广义特征值分别检索；在线性回归全局最优问题上，原本已命中第 36 页，拆分后仍保留必要页，说明没有把简单问题改坏。当前结果只证明这棵人工检查过的树在这些问题上有效，不能外推成任意问题都适合递归。

适用条件是问题本身包含多个相对独立的要求或多跳证据；单一事实题、强顺序推导题或上下文预算很紧时，拆分会增加检索和合并成本，甚至丢失限定条件。代码中的 `leaves` 展开树，`retrieve_tree` 为叶节点检索、按页去重，再把 `leaf_results` 留在每一步的输入和结果中；部署实现还应保存每个子答案和来源，合并时逐条检查条件。

In [3]:
from common.eval_utils import emit_tutorial_audit

# 统一保存契约：页码来自实际根问题/叶问题检索结果。
import json

def _actual_pages(items):
    pages = []
    for item in items:
        page = int(item.page)
        if page not in pages:
            pages.append(page)
    return pages

def _metrics(items, expected_pages):
    pages = _actual_pages(items)
    expected = {int(page) for page in expected_pages}
    found = set(pages) & expected
    rank = next((index for index, page in enumerate(pages, 1) if page in expected), None)
    return {'pages': pages, 'first_required_rank': rank,
            'required_page_coverage': len(found) / len(expected) if expected else 0.0}

def _emit(method, role, case_id, before_items, after_items, purpose=None):
    annotation = load_annotation(case_id)
    payload = {'case_id': case_id, 'method': method, 'role': role,
              'before': _metrics(before_items, annotation['expected_pages']),
              'after': _metrics(after_items, annotation['expected_pages'])}
    if purpose:
        payload['check_purpose'] = purpose
    emit_tutorial_audit(payload)

_emit('逐层拆分复杂问题（Recursive Decomposition）', 'main',
      'lda_derivation_recursive_steps', main_before, main_after)
_emit('逐层拆分复杂问题（Recursive Decomposition）', 'check',
      'linear_regression_global_optimum', check_before, check_after, '确认没有改坏')
